In [4]:
!git clone https://github.com/Bamboohoccode/SFT.git

Cloning into 'SFT'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 30 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 8.45 KiB | 2.11 MiB/s, done.
Resolving deltas: 100% (3/3), done.


In [6]:
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

/kaggle/working/SFT
Load file successfully


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Có thể tăng lên nếu RAM cho phép
dtype = None # Unsloth tự động chọn (thường là Float16 cho T4)
load_in_4bit = True 

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank, gợi ý: 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# Tải file dataset bạn đã upload lên Kaggle
dataset = load_dataset("json", data_files={"train": "/kaggle/input/ten-dataset-cua-ban/dataset.jsonl"}, split="train")

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3", # Sử dụng template chuẩn của Llama-3
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True,)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Pack nhiều sequence ngắn thành 1 sequence dài để train nhanh hơn
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Tăng số bước train tùy thuộc vào độ lớn của dataset
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# Bắt đầu quá trình huấn luyện
trainer_stats = trainer.train()

In [ ]:
# Inference (Sinh văn bản)
FastLanguageModel.for_inference(model) # Bật inference mode (nhanh gấp đôi)

messages = [
    {"role": "user", "content": "Kể cho tôi nghe về những cấu trúc dữ liệu bạn thường tìm hiểu đi."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Thêm token báo hiệu assistant chuẩn bị trả lời
    return_tensors = "pt",
).to("cuda")

# Sử dụng trực tiếp hàm generate của PyTorch model
outputs = model.generate(input_ids = inputs, max_new_tokens = 256, use_cache = True)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))